# Program Intake Triage Agent — Kaggle Demo

A multi-agent ADK pipeline that triages an enterprise program request and returns a
**human-reviewed recommendation** (never an automated approve/reject).

**Pipeline:** Intake Parser → Triage Scorer → Router / Capacity Check

This notebook clones the GitHub repo, loads the Gemini key from Kaggle Secrets, runs the
pipeline on a sample request, and applies the human-in-the-loop guardrail to the output.


## 1. Install dependencies


In [ ]:
!pip -q install google-adk google-genai pydantic


## 2. Get the code

Replace the URL with your public repo.


In [ ]:
import os, sys
REPO = 'program-intake-triage-agent'
if not os.path.exists(REPO):
    !git clone https://github.com/YOUR_USERNAME/program-intake-triage-agent.git
if os.path.basename(os.getcwd()) != REPO:
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())


## 3. Load your Gemini API key from Kaggle Secrets

Add-ons → Secrets → add a secret named `GOOGLE_API_KEY` with your AI Studio key.
This keeps the key out of the notebook and out of GitHub.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'FALSE'  # use AI Studio, not Vertex
print('Gemini key loaded:', bool(os.environ.get('GOOGLE_API_KEY')))


## 4. Build the runner


In [ ]:
import warnings, json, re
warnings.filterwarnings('ignore')  # clean demo output

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from intake_triage import root_agent

session_service = InMemorySessionService()
runner = Runner(agent=root_agent, app_name='intake_triage', session_service=session_service)

def _safe_json(text):
    """Parse a JSON string that may be wrapped in code fences."""
    if not text:
        return None
    cleaned = re.sub(r'^```(json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        return {'_raw': text}

def triage(request_text):
    """Run the pipeline and return the raw session state."""
    runner.run_debug(request_text, quiet=True)
    s = session_service.get_session_sync(app_name='intake_triage',
                                         user_id='debug_user_id', session_id='debug_session_id')
    return s.state
print('runner ready')


## 5. Run a sample request


In [ ]:
SAMPLE = (
    'Hi team, Sales wants a single view of each customer pulling together Salesforce '
    'records, support tickets, and product usage so reps can see churn risk. We think '
    'this is important and would like it within the quarter. Not sure who owns it.'
)

state = triage(SAMPLE)
parsed     = _safe_json(state.get('parsed_request'))
assessment = _safe_json(state.get('triage_assessment'))
routing    = _safe_json(state.get('routing_recommendation'))

print('PARSED:');       print(json.dumps(parsed, indent=2))
print('\nASSESSMENT:'); print(json.dumps(assessment, indent=2))
print('\nROUTING:');    print(json.dumps(routing, indent=2))


## 6. Apply the human-in-the-loop guardrail

This is the rule that defines the agent: it assembles a **recommendation** with a confidence
level and explicit review flags. It never emits an autonomous approve/reject.


In [ ]:
from intake_triage.skills.triage_rubric import score_to_tier
from intake_triage.guardrails import screen_request, enforce_human_in_the_loop

tier = score_to_tier(**assessment['scores']) if assessment and 'scores' in assessment else {}

flags = []
flags += screen_request(SAMPLE)['input_flags']
flags += (assessment or {}).get('missing_info', [])
flags += (routing or {}).get('open_questions', [])

final = enforce_human_in_the_loop(routing or {}, flags)
final['priority'] = tier

print(json.dumps(final, indent=2))


## Concepts demonstrated

| Course concept | Where |
|---|---|
| Multi-agent system (ADK) | `SequentialAgent` orchestrating 3 specialists |
| Sessions & state management | each agent's `output_key` -> session state, read by the next |
| Agent skills | the codified scoring rubric in `skills/triage_rubric.py` |
| Security / guardrail layer | input screening, tool-call guard, human-in-the-loop enforcement |
| MCP server integration *(optional upgrade)* | `mcp_server/portfolio_mcp_server.py` |

_The agent is advisory by design. A human reviewer makes every approve/reject decision._
